In [1]:
# train an autoencoder for 450K data
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

chr = 18
seed, epochs, batch_size, learning_rate = 0, 100, 128, 0.001

# Load the data, rows are features, columns are samples
data = pd.read_csv("../tmp/processed/GSE55763_chr{}.csv".format(chr), index_col="ID")
probe_names = data.index.values
sample_names = data.columns.values

# transpose the data so that rows are samples, columns are features
data = data.transpose()

# Handle NA values: Here, I'll replace NA with the mean of the column, but you can choose any other strategy.
data.fillna(data.mean(), inplace=True)

# split the data into training, validation, and testing sets
train_data = data.sample(frac=0.8, random_state=seed)
val_data = data.drop(train_data.index).sample(frac=0.5, random_state=seed)
test_data = data.drop(train_data.index).drop(val_data.index)

In [23]:
# generate probe bins, each bin contains 100 probes, bins overlap by 50 probes
probe_bins = []
for i in range(0, len(probe_names), 50):
    if i + 100 > len(probe_names):
        break
    probe_bins.append(probe_names[i:i + 100])

In [26]:
# for each bin, generate a training set
train_sets = []
for bin in probe_bins:
    train_sets.append(train_data[bin].to_numpy())
test_sets = []
for bin in probe_bins:
    test_sets.append(test_data[bin].to_numpy())
val_sets = []
for bin in probe_bins:
    val_sets.append(val_data[bin].to_numpy())

In [32]:
# concatenate the training sets, n_samples x 100, each row is a training sample
train_data = np.concatenate(train_sets, axis=0)
test_data = np.concatenate(test_sets, axis=0)
val_data = np.concatenate(val_sets, axis=0)

# convert the data to torch tensors, and add a channel dimension, n_samples x 1 x 100
train_data = torch.tensor(train_data, dtype=torch.float32).unsqueeze(1)
test_data = torch.tensor(test_data, dtype=torch.float32).unsqueeze(1)
val_data = torch.tensor(val_data, dtype=torch.float32).unsqueeze(1)

In [33]:
train_data.shape

torch.Size([253773, 1, 100])